# 00 — Context: Benefit Distribution as an Engineering Variable

**Repository:** `benefits-distribution`  
**Engineering statement:** **Benefit Distribution Specifies Sustainable Development**

This notebook is the entry point for a research program: treat benefit distribution as a measurable deployment state rather than a policy vibe.

The central claim is:

> Sustainable development requires more than generating capability. It requires measuring how deployed capability changes access, adoption, reliability, affordability, safety, and local capacity across populations.

Notebook 00 introduces four engineering objects:

1. a deployment pipeline,
2. a benefit-distribution state \(d_t\),
3. benefit telemetry \(E_t\),
4. a distribution-aware feedback loop.

## 1. From normative objective to engineering object

A paper can argue that technology should be globally beneficial.  
An engineering notebook asks a different question:

> What would have to be observed, estimated, and controlled for that objective to become operational?

We will use:

\[
d_t = G(P, T, E_t)
\]

where:

- \(P\) = population groups,
- \(T\) = technologies or capabilities,
- \(E_t\) = deployment telemetry,
- \(d_t\) = benefit-distribution state.

Sustainable-development indicators are then modeled as downstream functions of this state:

\[
D_t = f(d_t, A_t, R_t, C_t)
\]

where \(A_t\) is accessibility, \(R_t\) is reliability, and \(C_t\) is local capacity.

In [ ]:
# Setup

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

FIGURES = ROOT / "figures"
RESULTS = ROOT / "results"
DATA = ROOT / "data"

for path in [FIGURES, RESULTS, DATA]:
    path.mkdir(parents=True, exist_ok=True)

def savefig(name):
    path = FIGURES / name
    plt.savefig(path, dpi=220, bbox_inches="tight")
    print(f"saved: {path}")
    return path

print("ROOT:", ROOT)

## 2. System architecture

The important move is to place **benefit distribution** inside the technical loop.

Research produces capability.  
Capability becomes deployment.  
Deployment produces telemetry.  
Telemetry estimates a state.  
The state guides design and policy updates.

In [ ]:
# Figure 00_system_architecture.png

plt.figure(figsize=(13, 6))
ax = plt.gca()
ax.axis("off")

boxes = {
    "Research\nprogram": (0.08, 0.68),
    "Capability\nC_t": (0.28, 0.68),
    "Deployment\nπ_t": (0.48, 0.68),
    "Telemetry\nE_t": (0.68, 0.68),
    "Benefit distribution\nstate d_t": (0.48, 0.34),
    "Development\nindicators D_t": (0.82, 0.34),
    "Policy / design\nupdate": (0.20, 0.34),
}

for label, (x, y) in boxes.items():
    w = 0.15 if "Benefit" not in label else 0.20
    h = 0.18
    ax.add_patch(plt.Rectangle((x-w/2, y-h/2), w, h, fill=False, linewidth=1.8))
    ax.text(x, y, label, ha="center", va="center", fontsize=11, fontweight="bold" if "state" in label else None)

arrows = [
    ("Research\nprogram", "Capability\nC_t"),
    ("Capability\nC_t", "Deployment\nπ_t"),
    ("Deployment\nπ_t", "Telemetry\nE_t"),
    ("Telemetry\nE_t", "Benefit distribution\nstate d_t"),
    ("Benefit distribution\nstate d_t", "Development\nindicators D_t"),
    ("Benefit distribution\nstate d_t", "Policy / design\nupdate"),
    ("Policy / design\nupdate", "Deployment\nπ_t"),
]
for a, b in arrows:
    ax.annotate("", xy=boxes[b], xytext=boxes[a],
                arrowprops=dict(arrowstyle="->", linewidth=1.7, shrinkA=35, shrinkB=35))

ax.text(0.5, 0.95, "Benefit distribution as an observable deployment state",
        ha="center", fontsize=16, fontweight="bold")
ax.text(0.5, 0.08, "Engineering loop: deploy → observe → estimate → update → evaluate",
        ha="center", fontsize=12)

savefig("00_system_architecture.png")
plt.show()

## 3. Synthetic telemetry schema

For a first notebook, we do not need a real global dataset.  
We need a schema that makes the engineering target concrete.

Each population group receives a telemetry vector:

\[
E_{p,t} =
[\text{access}, \text{affordability}, \text{reliability}, \text{adoption}, \text{capacity}, \text{safety}]
\]

The synthetic values below are not claims about the world. They are placeholders for a measurement system.

In [ ]:
groups = ["rural clinics", "public schools", "small farms", "local labs", "municipal teams"]
metrics = ["access", "affordability", "reliability", "adoption", "local_capacity", "safety"]

telemetry = pd.DataFrame(
    [
        [0.61, 0.48, 0.72, 0.43, 0.39, 0.81],
        [0.74, 0.66, 0.69, 0.62, 0.51, 0.77],
        [0.52, 0.45, 0.64, 0.38, 0.47, 0.75],
        [0.83, 0.58, 0.79, 0.71, 0.76, 0.84],
        [0.68, 0.54, 0.73, 0.56, 0.63, 0.79],
    ],
    index=groups,
    columns=metrics,
)

telemetry_path = RESULTS / "00_synthetic_telemetry.csv"
telemetry.to_csv(telemetry_path)

telemetry

In [ ]:
# Figure 00_telemetry_matrix.png

plt.figure(figsize=(11, 5.5))
ax = plt.gca()
im = ax.imshow(telemetry.values, aspect="auto")

ax.set_xticks(np.arange(len(metrics)))
ax.set_xticklabels([m.replace("_", " ") for m in metrics], rotation=25, ha="right")
ax.set_yticks(np.arange(len(groups)))
ax.set_yticklabels(groups)

for i in range(telemetry.shape[0]):
    for j in range(telemetry.shape[1]):
        ax.text(j, i, f"{telemetry.values[i, j]:.2f}", ha="center", va="center", fontsize=9)

ax.set_title("Benefit telemetry matrix $E_t$", fontsize=15, fontweight="bold", pad=14)
ax.set_xlabel("Telemetry variable")
ax.set_ylabel("Population group")
cbar = plt.colorbar(im)
cbar.set_label("normalized signal")

savefig("00_telemetry_matrix.png")
plt.show()

## 4. Benefit-distribution graph

Telemetry can also be represented as a graph.

Population groups connect to technologies.  
Edges carry benefit weights.  
The graph is the state object:

\[
d_t = \{w_{p,k,t}\}
\]

This is the notebook's main technical hook: **benefit distribution becomes an observable network state.**

In [ ]:
# Figure 00_benefit_distribution_graph.png

populations = telemetry.index.tolist()
technologies = ["AI tools", "health tech", "energy systems", "open science"]

edge_weights = {
    ("rural clinics", "AI tools"): 0.52,
    ("rural clinics", "health tech"): 0.78,
    ("rural clinics", "energy systems"): 0.39,
    ("public schools", "AI tools"): 0.70,
    ("public schools", "open science"): 0.61,
    ("small farms", "energy systems"): 0.74,
    ("small farms", "AI tools"): 0.36,
    ("local labs", "open science"): 0.82,
    ("local labs", "AI tools"): 0.64,
    ("municipal teams", "AI tools"): 0.58,
    ("municipal teams", "energy systems"): 0.67,
    ("municipal teams", "open science"): 0.44,
}

G = nx.Graph()
for p in populations:
    G.add_node(p, kind="population")
for t in technologies:
    G.add_node(t, kind="technology")
for (u, v), w in edge_weights.items():
    G.add_edge(u, v, weight=w)

pos = {}
for i, p in enumerate(populations):
    pos[p] = (0, 1 - i/(len(populations)-1))
for i, t in enumerate(technologies):
    pos[t] = (1, 1 - i/(len(technologies)-1))

plt.figure(figsize=(11, 6.2))
ax = plt.gca()
ax.axis("off")

nx.draw_networkx_nodes(G, pos, nodelist=populations, node_size=2700, linewidths=1.8, edgecolors="black")
nx.draw_networkx_nodes(G, pos, nodelist=technologies, node_size=2700, linewidths=1.8, edgecolors="black")
nx.draw_networkx_labels(G, pos, font_size=9)

edge_widths = [1 + 5 * G.edges[e]["weight"] for e in G.edges]
nx.draw_networkx_edges(G, pos, width=edge_widths, alpha=0.75)

edge_labels = {(u, v): f"{d['weight']:.2f}" for u, v, d in G.edges(data=True)}
nx.draw_networkx_edge_labels(G, pos, edge_labels=edge_labels, font_size=8)

ax.text(0, 1.16, "Population groups", ha="center", fontsize=13, fontweight="bold")
ax.text(1, 1.16, "Technologies", ha="center", fontsize=13, fontweight="bold")
ax.text(0.5, -0.12, "Benefit-distribution state $d_t$: weighted population × technology graph",
        ha="center", fontsize=12)

savefig("00_benefit_distribution_graph.png")
plt.show()

## 5. Distribution-aware scoring

A useful score should not reward aggregate benefit while ignoring concentration.

A simple starter metric is:

\[
S = \bar{x}(1-\mathrm{CV})
\]

where:

- \(\bar{x}\) is mean observed benefit,
- \(\mathrm{CV}\) is coefficient of variation across groups.

This is deliberately simple. It marks the direction for later notebooks: define better estimators, not just prettier values.

In [ ]:
group_scores = telemetry.mean(axis=1)
mean_benefit = group_scores.mean()
cv = group_scores.std() / mean_benefit
distribution_score = mean_benefit * (1 - cv)

score_table = pd.DataFrame({
    "benefit_score": group_scores,
})
score_table.loc["mean"] = mean_benefit
score_table.loc["cv"] = cv
score_table.loc["distribution_aware_score"] = distribution_score

score_path = RESULTS / "00_distribution_scores.csv"
score_table.to_csv(score_path)

score_table

In [ ]:
# Figure 00_distribution_scores.png

plt.figure(figsize=(10, 4.8))
ax = plt.gca()
group_scores.plot(kind="bar", ax=ax)

ax.set_ylim(0, 1.0)
ax.set_ylabel("mean telemetry score")
ax.set_title("Group-level observed benefit scores", fontsize=15, fontweight="bold", pad=12)
ax.axhline(mean_benefit, linestyle="--", linewidth=1.5)
ax.text(len(group_scores)-0.5, mean_benefit + 0.02, f"mean = {mean_benefit:.2f}", ha="right")
plt.xticks(rotation=25, ha="right")

savefig("00_distribution_scores.png")
plt.show()

print("coefficient of variation:", round(cv, 3))
print("distribution-aware score:", round(distribution_score, 3))

## 6. Control loop

Once \(d_t\) is observable, the engineering agenda becomes:

\[
\pi_{t+1} \leftarrow \mathrm{Update}(\pi_t, d_t, D_t)
\]

where \(\pi_t\) is the deployment policy.

That update could mean changing:

- access conditions,
- pricing,
- infrastructure support,
- localization,
- safety constraints,
- training materials,
- open-source availability,
- or institutional partnerships.

In [ ]:
# Figure 00_control_loop.png

plt.figure(figsize=(9, 7))
ax = plt.gca()
ax.axis("off")

nodes = {
    "Deployment\npolicy π_t": (0.5, 0.86),
    "Capability\ndeployment": (0.82, 0.58),
    "Observed\ntelemetry E_t": (0.66, 0.22),
    "Benefit distribution\nstate d_t": (0.34, 0.22),
    "Policy / design\nupdate": (0.18, 0.58),
}

for label, (x, y) in nodes.items():
    ax.add_patch(plt.Circle((x, y), 0.12, fill=False, linewidth=1.8))
    ax.text(x, y, label, ha="center", va="center", fontsize=10)

edges = [
    ("Deployment\npolicy π_t", "Capability\ndeployment"),
    ("Capability\ndeployment", "Observed\ntelemetry E_t"),
    ("Observed\ntelemetry E_t", "Benefit distribution\nstate d_t"),
    ("Benefit distribution\nstate d_t", "Policy / design\nupdate"),
    ("Policy / design\nupdate", "Deployment\npolicy π_t"),
]

for a, b in edges:
    ax.annotate("", xy=nodes[b], xytext=nodes[a],
                arrowprops=dict(arrowstyle="->", linewidth=1.7, shrinkA=50, shrinkB=50))

ax.text(0.5, 0.04, r"$\pi_{t+1} \leftarrow \mathrm{Update}(\pi_t, d_t, D_t)$",
        ha="center", fontsize=13)
ax.text(0.5, 0.98, "Distribution-aware deployment control loop",
        ha="center", fontsize=15, fontweight="bold")

savefig("00_control_loop.png")
plt.show()

## 7. What Notebook 00 establishes

This notebook establishes the repo's basic grammar:

- **Capability** is not the same as **benefit**.
- Benefit requires telemetry.
- Telemetry estimates a state.
- The state can be represented as a graph.
- Concentration should be measured, not hand-waved.
- Deployment can be updated from observed distribution.

The next notebook should focus on \(d_t\) directly:

> **07 — Benefit Distribution as a State Variable**

## 8. Final download cell

Run the next cell after running the notebook. It creates a zip of generated outputs and displays download links for both the notebook and the outputs archive.

In [ ]:
# Final download cell

from pathlib import Path
from zipfile import ZipFile

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

NOTEBOOK = ROOT / "notebooks" / "00_context.ipynb"
ZIP = ROOT / "benefits-distribution-00-context-outputs.zip"

with ZipFile(ZIP, "w") as z:
    for folder_name in ["figures", "results", "data"]:
        folder = ROOT / folder_name
        if folder.exists():
            for file in folder.rglob("*"):
                if file.is_file():
                    z.write(file, file.relative_to(ROOT))

print("Created:", ZIP)

try:
    from IPython.display import FileLink, display
    if NOTEBOOK.exists():
        print("Notebook:")
        display(FileLink(str(NOTEBOOK)))
    print("Outputs:")
    display(FileLink(str(ZIP)))
except Exception:
    print("Notebook:", NOTEBOOK)
    print("Outputs:", ZIP)